In [ ]:
from pathlib import Path
import re

import pandas as pd
import geopandas as gpd
import os

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")

In [ ]:
# Lookup table from ensemble member ID to cost/damage sensitivity 
# parameters

ensemble_metadata = pd.read_csv("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/dphil_common_cross_cutting/common_incoming_data/networks/sensitivity_parameters.csv")

ensemble_metadata

In [ ]:
# CRS and affine transform parameters for each hazard raster file
# The affine transform parameters can be used to transform the grid
# indices in the damage files into lat/long

raster_metadata = pd.read_csv("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/dphil_common_cross_cutting/common_incoming_data/networks/_hazard_layers__with_transforms.csv")
raster_metadata.query('hazard == "fluvial" and rp == 1500 and rcp == "baseline"')

In [ ]:
raster_metadata.transform_id.unique()

In [ ]:
# Example damage file
def read_damage_file(fname):
    example_damage = pd.read_parquet(fname) 
    example_damage.head()
    to_drop_col_names = [
        col for col in example_damage.columns 
        if col.startswith("coastal") 
        or col.startswith("surface") 
        or (col.startswith("fluvial") and "baseline" not in col)
    ]
    to_drop_cell_index_cols = [
        col for col in example_damage.columns 
        if col.startswith("cell_index") 
        and "2" not in col
    ]
    example_damage = example_damage.drop(columns=to_drop_col_names+to_drop_cell_index_cols)
    
    return example_damage

example_damage = read_damage_file("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/dphil_common_cross_cutting/direct_damages_fred/damages/rail_edges_direct_damages_parameter_set_2.parquet").query('fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None > 0')
example_damage.columns, example_damage.shape

example_damage.head()

In [ ]:
# read non snapped points
non_snapped_points_with_flood = base_path / "dphil_paper_2/processed_data/delineating_upstream_catchments/non_snapped_flooded_asset_points/non_snapped_points_with_flood_indices.parquet"

non_snapped_points_with_flood = gpd.read_parquet(non_snapped_points_with_flood).reset_index().rename(columns={"index":"fid"})


non_snapped_points_with_flood.head()
# should keep all the damage rows, drop any flood points which didn't have railways

non_snapped_points_with_flood
#TO DO
# read snapped points
# merge on to non-snapped points using fid

### use pandas merge to merge the example_damage dataframe left_on flood_i, flood_j right_on cell_index_2_x, cell_index_2_y, how='right' but actually do the inverse - swap columns and do left


In [ ]:
# Keep all points (right), match damage on its cell_index_2_* to points’ flood_i/j
merged_non_snapped_points_with_flood_example_damage_df = example_damage.merge(
    non_snapped_points_with_flood,
    how="left",
    left_on=["cell_index_2_x", "cell_index_2_y"],
    right_on=["flood_i", "flood_j"],
)
unique_damage_fids = merged_non_snapped_points_with_flood_example_damage_df.dropna().fid.unique()
# example_damage[["cell_index_2_x", "cell_index_2_y"]].describe()
# non_snapped_points_with_flood[["flood_i", "flood_j"]].describe()
merged_non_snapped_points_with_flood_example_damage_df.head(2)

In [ ]:
snapped_points_with_dem = base_path / "dphil_paper_2/processed_data/delineating_upstream_catchments/snapped_flooded_assets_to_river/snapped_points_with_dem_indices.parquet"

snapped_points_with_dem = gpd.read_parquet(snapped_points_with_dem).reset_index().rename(columns={"index":"fid"})
snapped_points_with_dem.head()

snapped_points_with_dem

In [ ]:
merged_fid = non_snapped_points_with_flood.merge(
    snapped_points_with_dem,
    on="fid",
    how="left",
    validate="many_to_one"
)

In [ ]:
merged_fid.loc[sorted(unique_damage_fids)]

In [ ]:
subset = merged_fid.loc[sorted(unique_damage_fids)]


subset.to_parquet(base_path / "dphil_paper_2/processed_data/merged_fid_subset.parquet",
    index=False
)



In [ ]:
subset = merged_fid.loc[sorted(unique_damage_fids)]

save_path = base_path /"dphil_paper_2/processed_data/merged_fid_subset.parquet"


In [ ]:
points_to_catchment_df = merged_fid.loc[sorted(unique_damage_fids)][["geometry_y", "dem_i", "dem_j"]].drop_duplicates(subset=["dem_i","dem_j"])

In [ ]:
# save points_to_catchment_df to parquet
points_to_catchment_df.rename(columns={'geometry_y':'geometry'}, inplace=True)

In [ ]:
points_to_catchment_df.to_parquet(base_path / "dphil_paper_2/processed_data/points_to_catchment_new.parquet",
    index=False
)

points_to_catchment_df

In [ ]:
# # this cell for the other notebook

# for p in points_to_catchment_df.itertuples():
#     print(p.geometry_y.x, p.geometry_y.y) # easting, northing to get coordinates for GRASS r.water.outlet
#     print(p.dem_i, p.dem_j) # dem i and j to use in output name and filename (and later, reading in again here)
#     break

In [ ]:
# How to read every damage file in a loop, and pick out
# its asset class from the filename -- when you need to...


damage_dir = base_path / "dphil_papers/dphil_common_cross_cutting/direct_damages_fred/damages"

damage_paths = sorted(damage_dir.glob("*.parquet"))

# damage_paths = Path("damages").glob("*.parquet")

data = {}
for path in damage_paths:
    print(path)
    match = re.match(
        r"([A-Za-z0-9_.]+)_direct_damages_parameter_set_(\d+)\.parquet",
        Path(path).name
    )
    asset_class, ensemble_member = match.groups()
    # data[(asset_class, ensemble_member)] = read_damage_file(path)



damage_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/dphil_common_cross_cutting/direct_damages_fred/damages")
print("damage_dir:", damage_dir)
print("exists?", damage_dir.exists(), "is_dir?", damage_dir.is_dir())

files = list(damage_dir.glob("*.parquet"))
print("found", len(files), "parquet files")
for f in files:
    print(f.name)


In [ ]:
print(os.getcwd())